##**Assignment 3 (2024/2): ML1**
**Safe to eat or deadly poison?**



This homework is a classification task to identify whether a mushroom is edible or poisonous.

This dataset includes descriptions of hypothetical samples corresponding to 23 species of gilled mushrooms in the Agaricus and Lepiota Family Mushroom drawn from The Audubon Society Field Guide to North American Mushrooms (1981).

Each species is identified as definitely edible, definitely poisonous, or of unknown edibility and not recommended. This latter class was combined with the poisonous one. The Guide clearly states that there is no simple rule for determining the credibility of a mushroom; no rule like "leaflets three, let it be'' for Poisonous Oak and Ivy.


Step 1. Load 'mushroom2020_dataset.csv' data from the “Attachment” (note: this data set has been preliminarily prepared.).

Step 2. Drop rows where the target (label) variable is missing.

Step 3. Drop the following variables:
'id','gill-attachment', 'gill-spacing', 'gill-size','gill-color-rate', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring-rate','stalk-color-below-ring-rate','veil-color-rate','veil-type'

Step 4. Examine the number of rows, the number of digits, and whether any are missing.

Step 5. Fill missing values by adding the mean for numeric variables and the mode for nominal variables.

Step 6. Convert the label variable e (edible) to 1 and p (poisonous) to 0 and check the quantity. class0: class1

Step 7. Convert the nominal variable to numeric using a dummy code with drop_first = True.

Step 8. Split train/test with 20% test, stratify, and seed = 2020.

Step 9. Create a Random Forest with GridSearch on training data with 5 CV.
	'criterion':['gini','entropy']
'max_depth': [2,3]
'min_samples_leaf':[2,5]
'N_estimators':[100]
'random_state': 2020

Step 10.  Predict the testing data set with classification_report.


**Complete class MushroomClassifier from given code template below.**

In [46]:
#import your other libraries here
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
# hint
# import from sklearn.model_selection import train_test_split
# from sklearn.model_selection import ...
# from sklearn.ensemble import ...
# from sklearn.metrics import ...


In [ ]:
class MushroomClassifier:
    def __init__(self, data_path): # DO NOT modify this line
        self.data_path = data_path
        self.df = pd.read_csv(data_path)

    def Q1(self): # DO NOT modify this line
        """
            1. (From step 1) Before doing the data prep., how many "na" are there in "gill-size" variables?
        """
        return self.df['gill-size'].isnull().sum()


    def Q2(self): # DO NOT modify this line
        """
            2. (From step 2-4) How many rows of data, how many variables?
            - Drop rows where the target (label) variable is missing.
            - Drop the following variables:
            'id','gill-attachment', 'gill-spacing', 'gill-size','gill-color-rate','stalk-root', 'stalk-surface-above-ring',
            'stalk-surface-below-ring', 'stalk-color-above-ring-rate','stalk-color-below-ring-rate','veil-color-rate','veil-type'
            - Examine the number of rows, the number of digits, and whether any are missing.
        """
        self.df = self.df.dropna(subset=['label'])
        self.df = self.df.drop(['id','gill-attachment', 'gill-spacing', 'gill-size','gill-color-rate','stalk-root', 'stalk-surface-above-ring',
            'stalk-surface-below-ring', 'stalk-color-above-ring-rate','stalk-color-below-ring-rate','veil-color-rate','veil-type'], axis=1)
        return self.df.shape



    def Q3(self): # DO NOT modify this line
        """
            3. (From step 5-6) Answer the quantity class0:class1
            - Fill missing values by adding the mean for numeric variables and the mode for nominal variables.
            - Convert the label variable e (edible) to 1 and p (poisonous) to 0 and check the quantity. class0: class1
            - Note: You need to reproduce the process (code) from Q2 to obtain the correct result.
        """
        self.Q2()
        ncols = self.df.select_dtypes(include='number').columns
        ccols = self.df.select_dtypes(include='object').columns
        self.df = self.df.fillna(self.df.mean(numeric_only=True))
        for col in ccols:
            self.df[col] = self.df[col].fillna(self.df[col].mode().iloc[0])
        self.df['label'] = self.df['label'].apply(lambda x: 1 if x == 'e' else 0)
        return (self.df.loc[self.df['label'] == 0, 'label'].count(), self.df.loc[self.df['label'] == 1, 'label'].count())

    def Q4(self): # DO NOT modify this line
        """
            4. (From step 7-8) How much is each training and testing sets
            - Convert the nominal variable to numeric using a dummy code with drop_first = True.
            - Split train/test with 20% test, stratify, and seed = 2020.
            - Note: You need to reproduce the process (code) from Q2, Q3 to obtain the correct result.
        """
        self.Q3()
        self.df = pd.get_dummies(self.df, self.df.select_dtypes(include='object').columns, drop_first=True)
        y = self.df.pop("label")
        x  = self.df.copy()
        self.train_x, self.test_x, self.train_y, self.test_y = train_test_split(x, y, stratify=y, random_state=2020, test_size=0.2)
        return (self.train_x.shape, self.test_x.shape)

    def Q5(self):
        """
            5. (From step 9) Best params after doing random forest grid search.
            Create a Random Forest with GridSearch on training data with 5 CV.
            - 'criterion':['gini','entropy']
            - 'max_depth': [2,3]
            - 'min_samples_leaf':[2,5]
            - 'N_estimators':[100]
            - 'random_state': 2020
            - Note: You need to reproduce the process (code) from Q2, Q3, Q4 to obtain the correct result.
        """
        self.Q4()
        rd = RandomForestClassifier()
        params = {
            'criterion':['gini','entropy'],
            'max_depth': [2,3],
            'min_samples_leaf':[2,5],
            'n_estimators':[100],
            'random_state': [2020]
        }
        
        self.gs = GridSearchCV(estimator=rd, param_grid=params, n_jobs=-1, scoring='f1_weighted')
        self.gs.fit(self.train_x, self.train_y)
        return tuple(list(self.gs.best_params_.values()))

    def Q6(self):
        """
            5. (From step 10) What is the value of macro f1 (2 digits)?
            Predict the testing data set with confusion_matrix and classification_report,
            using scientific rounding (less than 0.5 dropped, more than 0.5 then increased)
            - Note: You need to reproduce the process (code) from Q2, Q3, Q4, Q5 to obtain the correct result.
        """
        self.Q5()
        predicts = self.gs.predict(self.test_x)
        reports = classification_report(self.test_y, predicts, output_dict=True)
        return (round(reports['0']['f1-score'], 2), round(reports['1']['f1-score'],2))


Run the code below to test that your code can work.

In [48]:
hw = MushroomClassifier('mushroom2020_dataset.csv')

print(hw.Q1())
# print(hw.Q2())
# print(hw.Q3())
# print(hw.Q4())
print(hw.Q5())
# print(hw.Q6())

121
('gini', 3, 2, 100, 2020)
